# SimpleCNN Baseline and Augmentation Experiments

This notebook tests a SimpleCNN for post-disaster building damage classification.

Classes:
- no-damage
- minor-damage
- major-damage
- destroyed

Experiments:
- Baseline
- Horizontal flip
- Rotation
- Color jitter
- Combined augmentation

Main observation:
The model has difficulty distinguishing minor damage from major damage.

## Results

Baseline: Macro F1 ≈ 0.49
Flip: Macro F1 ≈ 0.50
Rotation: Macro F1 ≈ 0.47
Color jitter: Macro F1 ≈ 0.48
Combined augmentation: Macro F1 ≈ 0.42

Horizontal flip performed best among the augmentation strategies tested.
Minor and major damage remain difficult for the model to distinguish.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
import os

print(os.listdir('/content/drive/MyDrive'))

['Jiayu Chen Resume.pdf', 'Colab Notebooks', 'Cropped Dataset']


In [5]:
IMG_DIR = "/content/drive/MyDrive/Cropped Dataset/train/post"

print(os.path.exists(IMG_DIR))

True


In [6]:
import pandas as pd

DATA_DIR = "/content/drive/MyDrive/Cropped Dataset"

manifest = pd.read_csv(
    DATA_DIR + "/manifest_train.csv"
)

print(manifest.shape)

(162787, 10)


In [7]:
label_map = {
    "no-damage":0,
    "minor-damage":1,
    "major-damage":2,
    "destroyed":3
}

manifest = manifest[manifest["damage_label"].isin(label_map.keys())]

print(manifest["damage_label"].value_counts())

damage_label
no-damage       117426
minor-damage     14980
major-damage     14161
destroyed        13227
Name: count, dtype: int64


In [8]:
from torch.utils.data import Dataset
from PIL import Image
import os
import torch

class DamagedDataset(Dataset):
    def __init__(self, manifest, img_dir, transform=None):
        self.manifest = manifest.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.manifest)

    def __getitem__(self, idx):
        row = self.manifest.iloc[idx]

        filename = os.path.basename(row["post_crop_path"])
        img_path = os.path.join(self.img_dir, filename)

        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        label = label_map[row["damage_label"]]

        return image, torch.tensor(label)

In [9]:
def check_file(row):
    filename = os.path.basename(row["post_crop_path"])
    return os.path.exists(os.path.join(IMG_DIR, filename))

manifest = manifest[manifest.apply(check_file, axis=1)]

print(len(manifest))

38812


In [10]:
from torchvision import transforms

transform = transforms.Compose([
    transforms.Resize((128,128)),
    transforms.ToTensor()
])

In [11]:
from torch.utils.data import Dataset
from PIL import Image
import os
import torch

class DamagedDataset(Dataset):
    def __init__(self, manifest, img_dir, transform=None):
        self.manifest = manifest.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.manifest)

    def __getitem__(self, idx):
        row = self.manifest.iloc[idx]

        filename = os.path.basename(row["post_crop_path"])
        img_path = os.path.join(self.img_dir, filename)

        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        label = label_map[row["damage_label"]]

        return image, torch.tensor(label)

In [12]:
import random
import numpy as np
import torch

seed = 42

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

print("Random seed fixed:", seed)

Random seed fixed: 42


In [13]:
dataset = DamagedDataset(
    manifest,
    IMG_DIR,
    transform=transform
)

from torch.utils.data import Subset

dataset = Subset(dataset, range(500))

print(len(dataset))

500


In [14]:
import pandas as pd

# Take 125 random samples from each damage class
balanced_manifest = (
    manifest.groupby("damage_label", group_keys=False)
    .sample(n=500, random_state=42)
    .reset_index(drop=True)
)

print(balanced_manifest["damage_label"].value_counts())
print("Total:", len(balanced_manifest))

damage_label
destroyed       500
major-damage    500
minor-damage    500
no-damage       500
Name: count, dtype: int64
Total: 2000


In [15]:
dataset = DamagedDataset(
    balanced_manifest,
    IMG_DIR,
    transform=transform
)

print("Dataset size:", len(dataset))

Dataset size: 2000


In [16]:
from torch.utils.data import DataLoader, random_split
import torch

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

generator = torch.Generator().manual_seed(42)

train_dataset, val_dataset = random_split(
    dataset,
    [train_size, val_size],
    generator=generator
)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=2
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2
)

print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))

Train: 1600
Validation: 400


In [17]:
import random
import numpy as np
import torch

seed = 42

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

if torch.cuda.is_available():
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

print("Random seed:", seed)

Random seed: 42


In [18]:
import torch
import torch.nn as nn

class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.model = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Flatten(),

            nn.Linear(64 * 16 * 16, 128),
            nn.ReLU(),

            nn.Linear(128, 4)
        )

    def forward(self, x):
        return self.model(x)

print("SimpleCNN defined")

SimpleCNN defined


In [19]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(device)

cpu


In [20]:
import torch.optim as optim

model = SimpleCNN().to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

print("Model ready")

Model ready


In [21]:
from torch.utils.data import DataLoader, random_split

In [22]:
train_size = int(0.8 * len(dataset))
val_size = len(dataset)-train_size

train_dataset, val_dataset = random_split(
    dataset,
    [train_size, val_size]
)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=2
)

In [23]:
images, labels = next(iter(train_loader))

print(images.shape)
print(labels.shape)

torch.Size([32, 3, 128, 128])
torch.Size([32])


In [24]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=2
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2
)

print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))

Train: 1600
Validation: 400


In [25]:
images, labels = next(iter(train_loader))

print(images.shape)
print(labels.shape)

torch.Size([32, 3, 128, 128])
torch.Size([32])


In [26]:
import torch.nn as nn

class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.model = nn.Sequential(
            nn.Conv2d(3,16,3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16,32,3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32,64,3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Flatten(),

            nn.Linear(64*16*16,128),
            nn.ReLU(),

            nn.Linear(128,4)
        )

    def forward(self,x):
        return self.model(x)

In [27]:
model = SimpleCNN().to(device)

print(model)

SimpleCNN(
  (model): Sequential(
    (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU()
    (8): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (9): Flatten(start_dim=1, end_dim=-1)
    (10): Linear(in_features=16384, out_features=128, bias=True)
    (11): ReLU()
    (12): Linear(in_features=128, out_features=4, bias=True)
  )
)


In [28]:
images, labels = next(iter(train_loader))

images = images.to(device)

output = model(images)

print(output.shape)

torch.Size([32, 4])


In [29]:
import torch.nn as nn
import torch.optim as optim

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

In [30]:
manifest = manifest[manifest["damage_label"].isin(label_map.keys())]

print(manifest["damage_label"].value_counts())
print(len(manifest))

damage_label
no-damage       28707
minor-damage     3560
major-damage     3367
destroyed        3178
Name: count, dtype: int64
38812


In [31]:
print(len(train_dataset))
print(len(val_dataset))
print(device)

1600
400
cpu


In [33]:
# Train CNN for 10 epochs
epochs = 10

for epoch in range(epochs):
    model.train()

    running_loss = 0

    for images, labels in train_loader:

        # move data to GPU
        images = images.to(device)
        labels = labels.to(device)

        # clear previous gradients
        optimizer.zero_grad()

        # forward pass
        outputs = model(images)

        # calculate loss
        loss = criterion(outputs, labels)

        # backward pass
        loss.backward()

        # update weights
        optimizer.step()

        running_loss += loss.item()

    avg_loss = running_loss / len(train_loader)

    print(
        f"Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}"
    )

KeyboardInterrupt: 

In [ ]:
epochs = 10

for epoch in range(epochs):

    model.train()
    running_loss = 0.0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    train_loss = running_loss / len(train_loader)

    model.eval()

    val_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(outputs, labels)
            val_loss += loss.item()

            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    val_loss = val_loss / len(val_loader)
    val_accuracy = 100 * correct / total

    print(
        f"Epoch {epoch+1}/{epochs} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_accuracy:.2f}%"
    )

In [36]:
model = SimpleCNN().to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

epochs = 10

In [ ]:
model.eval()

correct = 0
total = 0

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total

print(f"Validation Accuracy: {accuracy:.2f}%")

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
from collections import Counter

all_labels = []
all_preds = []

model.eval()

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(predicted.cpu().numpy())

print("True labels:", Counter(all_labels))
print("Predictions:", Counter(all_preds))

print("\nClassification Report:")
print(classification_report(
    all_labels,
    all_preds,
    labels=[0, 1, 2, 3],
    target_names=[
        "no-damage",
        "minor-damage",
        "major-damage",
        "destroyed"
    ],
    zero_division=0
))

print("Confusion Matrix:")
print(confusion_matrix(
    all_labels,
    all_preds,
    labels=[0, 1, 2, 3]
))

In [37]:
from torchvision import transforms

train_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(
        brightness=0.15,
        contrast=0.15
    ),
    transforms.ToTensor()
])

val_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor()
])

In [38]:
print(type(train_dataset))
print(type(val_dataset))

<class 'torch.utils.data.dataset.Subset'>
<class 'torch.utils.data.dataset.Subset'>


In [39]:
full_train_aug = DamagedDataset(
    balanced_manifest,
    IMG_DIR,
    transform=train_transform
)

full_val_aug = DamagedDataset(
    balanced_manifest,
    IMG_DIR,
    transform=val_transform
)

train_dataset_aug = torch.utils.data.Subset(
    full_train_aug,
    train_dataset.indices
)

val_dataset_aug = torch.utils.data.Subset(
    full_val_aug,
    val_dataset.indices
)

print(len(train_dataset_aug))
print(len(val_dataset_aug))

1600
400


In [40]:
train_loader_aug = DataLoader(
    train_dataset_aug,
    batch_size=32,
    shuffle=True,
    num_workers=2
)

val_loader_aug = DataLoader(
    val_dataset_aug,
    batch_size=32,
    shuffle=False,
    num_workers=2
)

In [41]:
model_aug = SimpleCNN().to(device)

criterion = nn.CrossEntropyLoss()

optimizer_aug = torch.optim.Adam(
    model_aug.parameters(),
    lr=0.001
)

In [42]:
epochs = 10

for epoch in range(epochs):
    model_aug.train()
    running_loss = 0.0

    for images, labels in train_loader_aug:
        images = images.to(device)
        labels = labels.to(device)

        optimizer_aug.zero_grad()

        outputs = model_aug(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer_aug.step()

        running_loss += loss.item()

    train_loss = running_loss / len(train_loader_aug)

    # Validation
    model_aug.eval()
    val_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in val_loader_aug:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model_aug(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item()

            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    val_loss = val_loss / len(val_loader_aug)
    val_accuracy = 100 * correct / total

    print(
        f"Epoch {epoch+1}/{epochs} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_accuracy:.2f}%"
    )

Epoch 1/10 | Train Loss: 1.3823 | Val Loss: 1.3518 | Val Acc: 35.50%
Epoch 2/10 | Train Loss: 1.3120 | Val Loss: 1.3106 | Val Acc: 34.75%
Epoch 3/10 | Train Loss: 1.2896 | Val Loss: 1.3492 | Val Acc: 31.75%
Epoch 4/10 | Train Loss: 1.3005 | Val Loss: 1.3095 | Val Acc: 35.25%
Epoch 5/10 | Train Loss: 1.2709 | Val Loss: 1.2943 | Val Acc: 40.50%
Epoch 6/10 | Train Loss: 1.2656 | Val Loss: 1.3114 | Val Acc: 42.25%
Epoch 7/10 | Train Loss: 1.2356 | Val Loss: 1.2618 | Val Acc: 42.00%
Epoch 8/10 | Train Loss: 1.2090 | Val Loss: 1.2415 | Val Acc: 46.50%
Epoch 9/10 | Train Loss: 1.1871 | Val Loss: 1.2336 | Val Acc: 46.50%
Epoch 10/10 | Train Loss: 1.1500 | Val Loss: 1.1978 | Val Acc: 47.00%


In [44]:
from sklearn.metrics import classification_report, confusion_matrix
from collections import Counter

all_labels = []
all_preds = []

model_aug.eval()

with torch.no_grad():
    for images, labels in val_loader_aug:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model_aug(images)
        _, predicted = torch.max(outputs, 1)

        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(predicted.cpu().numpy())

print("True labels:", Counter(all_labels))
print("Predictions:", Counter(all_preds))

print("\nClassification Report:")
print(classification_report(
    all_labels,
    all_preds,
    labels=[0, 1, 2, 3],
    target_names=[
        "no-damage",
        "minor-damage",
        "major-damage",
        "destroyed"
    ],
    zero_division=0
))

print("Confusion Matrix:")
print(confusion_matrix(
    all_labels,
    all_preds,
    labels=[0, 1, 2, 3]
))

True labels: Counter({np.int64(2): 112, np.int64(0): 106, np.int64(1): 95, np.int64(3): 87})
Predictions: Counter({np.int64(0): 158, np.int64(3): 93, np.int64(2): 87, np.int64(1): 62})

Classification Report:
              precision    recall  f1-score   support

   no-damage       0.42      0.63      0.51       106
minor-damage       0.42      0.27      0.33        95
major-damage       0.55      0.43      0.48       112
   destroyed       0.51      0.54      0.52        87

    accuracy                           0.47       400
   macro avg       0.48      0.47      0.46       400
weighted avg       0.48      0.47      0.46       400

Confusion Matrix:
[[67 11 16 12]
 [33 26 18 18]
 [36 12 48 16]
 [22 13  5 47]]


In [45]:
from torchvision import transforms
from torch.utils.data import DataLoader
import torch
import torch.nn as nn
import torch.optim as optim

# 1. Mild augmentation: ONLY horizontal flip
flip_train_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor()
])

flip_val_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor()
])

# 2. Use exactly the same data and exactly the same split
full_train_flip = DamagedDataset(
    balanced_manifest,
    IMG_DIR,
    transform=flip_train_transform
)

full_val_flip = DamagedDataset(
    balanced_manifest,
    IMG_DIR,
    transform=flip_val_transform
)

train_dataset_flip = torch.utils.data.Subset(
    full_train_flip,
    train_dataset.indices
)

val_dataset_flip = torch.utils.data.Subset(
    full_val_flip,
    val_dataset.indices
)

# 3. DataLoaders
train_loader_flip = DataLoader(
    train_dataset_flip,
    batch_size=32,
    shuffle=True,
    num_workers=2
)

val_loader_flip = DataLoader(
    val_dataset_flip,
    batch_size=32,
    shuffle=False,
    num_workers=2
)

print("Train:", len(train_dataset_flip))
print("Validation:", len(val_dataset_flip))

# 4. Fresh model
model_flip = SimpleCNN().to(device)

criterion = nn.CrossEntropyLoss()

optimizer_flip = optim.Adam(
    model_flip.parameters(),
    lr=0.001
)

# 5. Train for 10 epochs
epochs = 10

for epoch in range(epochs):
    model_flip.train()
    running_loss = 0.0

    for images, labels in train_loader_flip:
        images = images.to(device)
        labels = labels.to(device)

        optimizer_flip.zero_grad()

        outputs = model_flip(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer_flip.step()

        running_loss += loss.item()

    train_loss = running_loss / len(train_loader_flip)

    # validation
    model_flip.eval()
    val_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in val_loader_flip:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model_flip(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item()

            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    val_loss = val_loss / len(val_loader_flip)
    val_accuracy = 100 * correct / total

    print(
        f"Epoch {epoch+1}/{epochs} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_accuracy:.2f}%"
    )

Train: 1600
Validation: 400
Epoch 1/10 | Train Loss: 1.3758 | Val Loss: 1.3674 | Val Acc: 34.50%
Epoch 2/10 | Train Loss: 1.3106 | Val Loss: 1.3139 | Val Acc: 37.00%
Epoch 3/10 | Train Loss: 1.3041 | Val Loss: 1.3294 | Val Acc: 34.75%
Epoch 4/10 | Train Loss: 1.2793 | Val Loss: 1.2973 | Val Acc: 40.50%
Epoch 5/10 | Train Loss: 1.2650 | Val Loss: 1.2793 | Val Acc: 40.00%
Epoch 6/10 | Train Loss: 1.2486 | Val Loss: 1.2838 | Val Acc: 37.50%
Epoch 7/10 | Train Loss: 1.2312 | Val Loss: 1.2827 | Val Acc: 40.25%
Epoch 8/10 | Train Loss: 1.2031 | Val Loss: 1.2870 | Val Acc: 39.50%
Epoch 9/10 | Train Loss: 1.1880 | Val Loss: 1.3637 | Val Acc: 40.75%
Epoch 10/10 | Train Loss: 1.1896 | Val Loss: 1.2722 | Val Acc: 41.00%


In [46]:
from sklearn.metrics import classification_report, confusion_matrix
from collections import Counter

all_labels_flip = []
all_preds_flip = []

model_flip.eval()

with torch.no_grad():
    for images, labels in val_loader_flip:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model_flip(images)
        _, predicted = torch.max(outputs, 1)

        all_labels_flip.extend(labels.cpu().numpy())
        all_preds_flip.extend(predicted.cpu().numpy())

print("True labels:", Counter(all_labels_flip))
print("Predictions:", Counter(all_preds_flip))

print("\nClassification Report:")
print(classification_report(
    all_labels_flip,
    all_preds_flip,
    labels=[0, 1, 2, 3],
    target_names=[
        "no-damage",
        "minor-damage",
        "major-damage",
        "destroyed"
    ],
    zero_division=0
))

print("Confusion Matrix:")
print(confusion_matrix(
    all_labels_flip,
    all_preds_flip,
    labels=[0, 1, 2, 3]
))

True labels: Counter({np.int64(2): 112, np.int64(0): 106, np.int64(1): 95, np.int64(3): 87})
Predictions: Counter({np.int64(2): 159, np.int64(0): 91, np.int64(1): 79, np.int64(3): 71})

Classification Report:
              precision    recall  f1-score   support

   no-damage       0.35      0.30      0.32       106
minor-damage       0.38      0.32      0.34        95
major-damage       0.42      0.60      0.49       112
   destroyed       0.49      0.40      0.44        87

    accuracy                           0.41       400
   macro avg       0.41      0.40      0.40       400
weighted avg       0.41      0.41      0.40       400

Confusion Matrix:
[[32 20 43 11]
 [19 30 33 13]
 [15 18 67 12]
 [25 11 16 35]]


In [47]:
from torchvision import transforms
from torch.utils.data import DataLoader
import torch
import torch.nn as nn
import torch.optim as optim

rotation_train_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.RandomRotation(10),
    transforms.ToTensor()
])

rotation_val_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor()
])

full_train_rotation = DamagedDataset(
    balanced_manifest,
    IMG_DIR,
    transform=rotation_train_transform
)

full_val_rotation = DamagedDataset(
    balanced_manifest,
    IMG_DIR,
    transform=rotation_val_transform
)

train_dataset_rotation = torch.utils.data.Subset(
    full_train_rotation,
    train_dataset.indices
)

val_dataset_rotation = torch.utils.data.Subset(
    full_val_rotation,
    val_dataset.indices
)

train_loader_rotation = DataLoader(
    train_dataset_rotation,
    batch_size=32,
    shuffle=True,
    num_workers=2
)

val_loader_rotation = DataLoader(
    val_dataset_rotation,
    batch_size=32,
    shuffle=False,
    num_workers=2
)

model_rotation = SimpleCNN().to(device)

criterion = nn.CrossEntropyLoss()

optimizer_rotation = optim.Adam(
    model_rotation.parameters(),
    lr=0.001
)

epochs = 10

for epoch in range(epochs):
    model_rotation.train()
    running_loss = 0.0

    for images, labels in train_loader_rotation:
        images = images.to(device)
        labels = labels.to(device)

        optimizer_rotation.zero_grad()

        outputs = model_rotation(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer_rotation.step()

        running_loss += loss.item()

    train_loss = running_loss / len(train_loader_rotation)

    model_rotation.eval()
    val_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in val_loader_rotation:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model_rotation(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item()

            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    val_loss = val_loss / len(val_loader_rotation)
    val_accuracy = 100 * correct / total

    print(
        f"Epoch {epoch+1}/{epochs} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_accuracy:.2f}%"
    )

Epoch 1/10 | Train Loss: 1.3803 | Val Loss: 1.3437 | Val Acc: 30.00%
Epoch 2/10 | Train Loss: 1.3124 | Val Loss: 1.3309 | Val Acc: 34.75%
Epoch 3/10 | Train Loss: 1.2759 | Val Loss: 1.3278 | Val Acc: 36.25%
Epoch 4/10 | Train Loss: 1.2550 | Val Loss: 1.3007 | Val Acc: 41.25%
Epoch 5/10 | Train Loss: 1.2542 | Val Loss: 1.2575 | Val Acc: 40.50%
Epoch 6/10 | Train Loss: 1.2093 | Val Loss: 1.2485 | Val Acc: 42.75%
Epoch 7/10 | Train Loss: 1.1856 | Val Loss: 1.2545 | Val Acc: 42.50%
Epoch 8/10 | Train Loss: 1.1589 | Val Loss: 1.2705 | Val Acc: 41.00%
Epoch 9/10 | Train Loss: 1.1461 | Val Loss: 1.2252 | Val Acc: 46.00%
Epoch 10/10 | Train Loss: 1.0780 | Val Loss: 1.2348 | Val Acc: 46.25%


In [48]:
from sklearn.metrics import classification_report, confusion_matrix

all_labels_rotation = []
all_preds_rotation = []

model_rotation.eval()

with torch.no_grad():
    for images, labels in val_loader_rotation:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model_rotation(images)
        _, predicted = torch.max(outputs, 1)

        all_labels_rotation.extend(labels.cpu().numpy())
        all_preds_rotation.extend(predicted.cpu().numpy())

print(classification_report(
    all_labels_rotation,
    all_preds_rotation,
    labels=[0, 1, 2, 3],
    target_names=[
        "no-damage",
        "minor-damage",
        "major-damage",
        "destroyed"
    ],
    zero_division=0
))

print("Confusion Matrix:")
print(confusion_matrix(
    all_labels_rotation,
    all_preds_rotation,
    labels=[0, 1, 2, 3]
))

              precision    recall  f1-score   support

   no-damage       0.46      0.42      0.44       106
minor-damage       0.36      0.47      0.41        95
major-damage       0.56      0.40      0.47       112
   destroyed       0.51      0.57      0.54        87

    accuracy                           0.46       400
   macro avg       0.47      0.47      0.47       400
weighted avg       0.48      0.46      0.46       400

Confusion Matrix:
[[45 31 15 15]
 [13 45 16 21]
 [22 33 45 12]
 [17 15  5 50]]


In [49]:
color_train_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ColorJitter(
        brightness=0.15,
        contrast=0.15
    ),
    transforms.ToTensor()
])

color_val_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor()
])

full_train_color = DamagedDataset(
    balanced_manifest,
    IMG_DIR,
    transform=color_train_transform
)

full_val_color = DamagedDataset(
    balanced_manifest,
    IMG_DIR,
    transform=color_val_transform
)

train_dataset_color = torch.utils.data.Subset(
    full_train_color,
    train_dataset.indices
)

val_dataset_color = torch.utils.data.Subset(
    full_val_color,
    val_dataset.indices
)

train_loader_color = DataLoader(
    train_dataset_color,
    batch_size=32,
    shuffle=True,
    num_workers=2
)

val_loader_color = DataLoader(
    val_dataset_color,
    batch_size=32,
    shuffle=False,
    num_workers=2
)

model_color = SimpleCNN().to(device)

criterion = nn.CrossEntropyLoss()

optimizer_color = optim.Adam(
    model_color.parameters(),
    lr=0.001
)

epochs = 10

for epoch in range(epochs):
    model_color.train()
    running_loss = 0.0

    for images, labels in train_loader_color:
        images = images.to(device)
        labels = labels.to(device)

        optimizer_color.zero_grad()

        outputs = model_color(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer_color.step()

        running_loss += loss.item()

    train_loss = running_loss / len(train_loader_color)

    model_color.eval()
    correct = 0
    total = 0
    val_loss = 0.0

    with torch.no_grad():
        for images, labels in val_loader_color:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model_color(images)

            loss = criterion(outputs, labels)
            val_loss += loss.item()

            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    val_loss = val_loss / len(val_loader_color)
    val_accuracy = 100 * correct / total

    print(
        f"Epoch {epoch+1}/{epochs} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_accuracy:.2f}%"
    )

Epoch 1/10 | Train Loss: 1.3851 | Val Loss: 1.3506 | Val Acc: 36.50%
Epoch 2/10 | Train Loss: 1.3202 | Val Loss: 1.3375 | Val Acc: 33.50%
Epoch 3/10 | Train Loss: 1.2947 | Val Loss: 1.3156 | Val Acc: 37.25%
Epoch 4/10 | Train Loss: 1.2622 | Val Loss: 1.2542 | Val Acc: 39.50%
Epoch 5/10 | Train Loss: 1.2350 | Val Loss: 1.2759 | Val Acc: 43.00%
Epoch 6/10 | Train Loss: 1.1925 | Val Loss: 1.2459 | Val Acc: 44.00%
Epoch 7/10 | Train Loss: 1.1618 | Val Loss: 1.1848 | Val Acc: 50.00%
Epoch 8/10 | Train Loss: 1.1196 | Val Loss: 1.2292 | Val Acc: 48.50%
Epoch 9/10 | Train Loss: 1.0596 | Val Loss: 1.2014 | Val Acc: 46.25%
Epoch 10/10 | Train Loss: 1.0367 | Val Loss: 1.2553 | Val Acc: 48.25%


In [50]:
all_labels_color = []
all_preds_color = []

model_color.eval()

with torch.no_grad():
    for images, labels in val_loader_color:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model_color(images)
        _, predicted = torch.max(outputs, 1)

        all_labels_color.extend(labels.cpu().numpy())
        all_preds_color.extend(predicted.cpu().numpy())

print(classification_report(
    all_labels_color,
    all_preds_color,
    labels=[0, 1, 2, 3],
    target_names=[
        "no-damage",
        "minor-damage",
        "major-damage",
        "destroyed"
    ],
    zero_division=0
))

print("Confusion Matrix:")
print(confusion_matrix(
    all_labels_color,
    all_preds_color,
    labels=[0, 1, 2, 3]
))

              precision    recall  f1-score   support

   no-damage       0.45      0.61      0.52       106
minor-damage       0.38      0.39      0.39        95
major-damage       0.58      0.38      0.46       112
   destroyed       0.57      0.56      0.57        87

    accuracy                           0.48       400
   macro avg       0.50      0.49      0.48       400
weighted avg       0.50      0.48      0.48       400

Confusion Matrix:
[[65 23  9  9]
 [25 37 17 16]
 [30 28 42 12]
 [25  9  4 49]]
